# Review Base Rag

In [2]:
# Set up the OpenAI client
from dotenv import  load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
# Load date and build the search engine
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

In [4]:
# Create the assistant
assistant = RAGBase(
    index=index,
    llm_client=openai_client)


In [5]:
assistant.rag("How do I run Ollama locally?")

'Run Ollama locally by first installing it from https://ollama.com/download for your OS, then in a terminal run:\n\n```bash\nollama run llama3\n```\n\nThis downloads the model, starts it locally, and opens a chat interface.\n\nTo check that the local server is running, you can also try:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n\n```bash\npip install ollama\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

"I don't know."

# Agentic Rag

In [7]:
message = [{"role": "user", "content": "I just discovered the course. Can I join it?"}]

response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message
)

response.output_text

'Yes, probably — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Just send me:\n- the course name\n- the school/platform\n- whether it’s online or in-person\n- any deadlines you’ve seen\n\nIf you’re asking what to say to the instructor or support team, you could message:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll? If so, could you please let me know the next steps?\n\nIf you want, I can also help you write a more formal or more casual message.'

In [8]:
import json

print(response.model_dump_json(indent=2))

{
  "id": "resp_0973293854e52b25006a360ccd23dc819e9f46d3cda9442fc7",
  "created_at": 1781927117.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0973293854e52b25006a360ccdb658819e9cb43a3c947a6530",
      "content": [
        {
          "annotations": [],
          "text": "Yes, probably — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Just send me:\n- the course name\n- the school/platform\n- whether it’s online or in-person\n- any deadlines you’ve seen\n\nIf you’re asking what to say to the instructor or support team, you could message:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll? If so, could you please let me know the next steps?\n\nIf you want, I can also help you write a more formal or 

In [9]:
def search(query):
    boost_dic = {"question": 3.0, "section": 0.5}
    filter_dic = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dic,
        filter_dict=filter_dic
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message,
    tools = [search_tool]
)   

response.output

[ResponseFunctionToolCall(arguments='{"query":"can I join course late discovered just discovered enrollment join after start"}', call_id='call_Mdh14ok2bfonjihsHre0oRYw', name='search', type='function_call', id='fc_0005831f8003afa9006a360ccfa110819e89a8da293c64496f', namespace=None, status='completed')]

In [12]:
response.output_text

''

In [13]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0005831f8003afa9006a360ccef8ac819e86ee2a6d1db44a05",
  "created_at": 1781927118.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "arguments": "{\"query\":\"can I join course late discovered just discovered enrollment join after start\"}",
      "call_id": "call_Mdh14ok2bfonjihsHre0oRYw",
      "name": "search",
      "type": "function_call",
      "id": "fc_0005831f8003afa9006a360ccfa110819e89a8da293c64496f",
      "namespace": null,
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "search",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Search query text to look up in the course FAQ."
          }
        },
        "required": [
          

In [14]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [15]:
message.extend(response.output)
message.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
})

In [16]:
response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message,
    tools = [search_tool]
)
response.output_text

'Yes — you can still join and start learning now.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'

In [17]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(773, 34)

In [18]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Agentic Loop

## A developer prompt


In [19]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## A function-call helper

In [20]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        results = search(**args)

    results_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": results_json
    }



In [21]:
question = "I just discovered the course. Can I join it?"

message = [{"role": "developer", "content": instructions},
           {"role": "user", "content": question}]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message,
    tools=[search_tool]
)

message.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        message.append(call_output)
        has_function_calls = True

    elif item.type  == "message":
        print("ASSISTANT:")
        print(item.content[0].text)


function_call: search {"query":"join course discovered course can I join enrollment late join"}
function_call: search {"query":"course FAQ enrollment register join after start late admission"}


## The full agent loop

In [22]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=message,
        tools=[search_tool]
    )


    message.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            message.append(call_output)
            has_function_calls = True

        elif item.type  == "message":
            print("ASSISTANT:")
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

You can start learning and submitting homework while the forms are open, and registration is not required to begin. If you want a certificate, though, you need to complete the project and submit it while the course is still accepting submissions.

If you'd like, I can also help you figure out how to start or what the weekly workflow looks like.


## Wrapping it in a function


In [29]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            print("ITEM TYPE:", item.type)
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            
            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)
            
        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [30]:
agent_loop(instructions, "How do I run Olama locally?")


iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"Olama local run install run locally"}
iteration #2...
ITEM TYPE: message
ASSISTANT:
I found the Ollama install/run instructions in the course FAQ, but I’ll do one more search to confirm the local server and Python client details.
ITEM TYPE: function_call
function_call: search {"query":"ollama localhost 11434 python client chat model llama3 course FAQ"}
iteration #3...
ITEM TYPE: message
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat-like prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
 

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat-like prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should see a response indicating Ollama is available.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, restarting the server with:\n\n```bash\nollama serve\n```\n\nca

## Encouraging multiple searches

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

You must use the search function before answering.
You must perform at least two sequential searches.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"join course discovered course can I join enrollment registration"}
iteration #2...
ITEM TYPE: function_call
function_call: search {"query":"self paced certificate live cohort peer review submit project accepted while form open" }
iteration #3...
ITEM TYPE: message
ASSISTANT:
Yes — you can still join the course. The FAQ says that if you just discovered it, you can still start learning and participate.

One important caveat: if you want a certificate, you need to submit your project while submissions are still open, and certificates are only available in the live cohort, not self-paced mode.

If you want, I can also help you figure out how to start the course or what the weekly workflow looks like. Is there anything else you’d like to explore?


'Yes — you can still join the course. The FAQ says that if you just discovered it, you can still start learning and participate.\n\nOne important caveat: if you want a certificate, you need to submit your project while submissions are still open, and certificates are only available in the live cohort, not self-paced mode.\n\nIf you want, I can also help you figure out how to start the course or what the weekly workflow looks like. Is there anything else you’d like to explore?'

## Restricting off-topic questions


In [32]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"queen gambit"}
iteration #2...
ITEM TYPE: function_call
function_call: search {"query":"queen's gambit opening chess"}
iteration #3...
ITEM TYPE: function_call
function_call: search {"query":"chess opening queen's gambit definition"}
iteration #4...
ITEM TYPE: message
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening** that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to distract Black’s **d-pawn** and gain control of the center. It’s called a “gambit” because White is offering a pawn, though in many lines the pawn is not truly lost.

A common response is:
- **Queen’s Gambit Declined**: Black does **not** take the c-pawn
- **Queen’s Gambit Accepted**: Black **does** take the c-pawn

If you want, I can also explain:
- the main ideas behind the opening,
- the difference between accepted and declined,
- or show a simple move-by-move example.

Is there another area you want to 

'The **Queen’s Gambit** is a classic **chess opening** that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to distract Black’s **d-pawn** and gain control of the center. It’s called a “gambit” because White is offering a pawn, though in many lines the pawn is not truly lost.\n\nA common response is:\n- **Queen’s Gambit Declined**: Black does **not** take the c-pawn\n- **Queen’s Gambit Accepted**: Black **does** take the c-pawn\n\nIf you want, I can also explain:\n- the main ideas behind the opening,\n- the difference between accepted and declined,\n- or show a simple move-by-move example.\n\nIs there another area you want to explore?'

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
ITEM TYPE: function_call
function_call: search {"query":"queen gambit"}
iteration #2...
ITEM TYPE: function_call
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ITEM TYPE: message
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so I’m not able to answer that from the course materials.

If you meant a course-specific term or logistic, feel free to rephrase it, and I can check again. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so I’m not able to answer that from the course materials.\n\nIf you meant a course-specific term or logistic, feel free to rephrase it, and I can check again. Are there other areas you want to explore?'